<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week02_Intelligent_Agents_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 2 Practical (PH):** Intelligent agents — PEAS, rationality, reflex vs model-based

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO2

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — The vacuum world (task environment) | Task A1 — Build a **model-based** agent |
| Part B — Is it *rational*? Evaluate with statistics | Task B1 — Classify task environments<br>Task B3 — Score the agent and update its belief (note §2.2 and §2.5)<br>Task B2 — Agent types and architectures |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | write a **PEAS** description and classify a task environment on six dimensions | Apply | LO2 |
| 2 | implement random, simple-reflex and **model-based** agents | Apply | LO2 |
| 3 | define a performance measure and judge **rationality** as *expected* performance | Evaluate | LO2 |
| 4 | compare agents with means, 95% CIs and a paired non-parametric test | Evaluate | LO2 |

**Formal view.** An agent is a function from percept histories to actions, $f: \mathcal{P}^{*} \rightarrow \mathcal{A}$. The *agent program* is the code that implements $f$. A **rational agent** picks the action that maximises the *expected* value of the performance measure given the percepts so far:
$$a^{*} = \arg\max_{a \in \mathcal{A}} \; \mathbb{E}\big[\,U \mid \text{percept history},\, a\,\big].$$

**Do Now (3 min):** write a PEAS for a campus cleaning robot at OOU (Performance, Environment, Actuators, Sensors).

## Part A — The vacuum world (task environment)
A 5×5 grid. Each cell is dirty with probability 0.3. **Percept** each step: `dirty` (is my cell dirty?) and `bump` (did my last move hit a wall?). **Actions:** `UP, DOWN, LEFT, RIGHT, CLEAN, NOOP`.
**Performance measure:** +10 for cleaning a dirty cell, −1 for every move or useless clean, 0 for `NOOP`. Episode length: 100 steps.

### 🔍 Code walkthrough — the sense–act loop
An agent never sees the whole world. The environment hands it a **percept** (is my square dirty? did I bump into a wall?), the agent returns an **action**, and the environment updates the world and the score. That loop is `run_episode`.

Notice three design choices: the environment has its **own** random generator (so the experiment is reproducible), the agent has a **different** generator (so it cannot "cheat" by reading the environment's randomness), and the score rewards cleaning but charges one point per move (the energy penalty).

In [ ]:
#@title 🧪 Environment (seeded, self-contained RNG)
# The six actions the vacuum agent may choose
ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT", "CLEAN", "NOOP"]

# The environment: an n x n grid, some cells dirty
class GridWorld:
    # Constructor: size, probability a cell is dirty, random seed
    def __init__(self, n=5, dirt_prob=0.3, seed=0):
        # The environment owns its own random generator, so runs are reproducible
        self.rng = np.random.default_rng(seed)               # own RNG → no hidden global state
        # Remember the grid size
        self.n = n
        # Each cell is dirty (1) with probability dirt_prob, otherwise clean (0)
        self.dirt = (self.rng.random((n, n)) < dirt_prob).astype(int)
        # Agent starts top-left with score 0 and no bump
        self.pos = (0, 0); self.score = 0; self.bump = False

    # What the agent is allowed to see (a percept), not the whole grid
    def percept(self):
        # Percept = is my current cell dirty? did I just hit a wall?
        return {"dirty": bool(self.dirt[self.pos]), "bump": self.bump}

    # Apply one action and update the world
    def step(self, action):
        # Current position; reset the bump flag
        x, y = self.pos; self.bump = False
        # Cleaning action
        if action == "CLEAN":
            # If the cell was dirty, clean it and gain 10 points
            if self.dirt[x, y]: self.dirt[x, y] = 0; self.score += 10
            # Cleaning a clean cell wastes effort: -1
            else:               self.score -= 1
        # Movement actions
        elif action in ("UP", "DOWN", "LEFT", "RIGHT"):
            # Row/column change for each direction
            dx, dy = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}[action]
            # Where the agent would end up
            nx, ny = x + dx, y + dy
            # Move only if the new cell is inside the grid
            if 0 <= nx < self.n and 0 <= ny < self.n: self.pos = (nx, ny)
            # Otherwise the agent bumps into the wall
            else: self.bump = True                           # walked into a wall
            # Every move costs 1 point (the energy penalty)
            self.score -= 1
        # Any other action except NOOP is illegal
        elif action != "NOOP":
            # Penalise illegal actions
            self.score -= 1                                  # illegal action
        # Return the new percept to the agent
        return self.percept()

def run_episode(agent_factory, seed, steps=100):
    """Fresh world + fresh agent per episode. Same seed ⇒ same dirt layout for every agent (paired design)."""
    # Create a fresh world for this episode
    env = GridWorld(seed=seed)
    # Build an agent with its own random generator
    agent = agent_factory(np.random.default_rng(10_000 + seed))   # agent's own RNG
    # First percept before any action
    p = env.percept()
    # Run the sense-act loop for a fixed number of steps
    for _ in range(steps):
        # The agent chooses an action from its percept; the environment executes it
        p = env.step(agent(p))
    # Return the final score and how many cells remain dirty
    return env.score, int(env.dirt.sum())

# --- two baseline agents (stateless: they only see the current percept) ---------------------
# Baseline agent that ignores its percept
def random_agent(rng):
    # Pick one of the first five actions at random
    return lambda p: ACTIONS[rng.integers(0, 5)]               # any of the 5 non-NOOP actions

# Simple reflex agent: acts only on the current percept
def reflex_agent(rng):
    # The decision function the environment will call
    def act(p):
        # Condition-action rule: if dirty then clean
        if p["dirty"]: return "CLEAN"                          # condition–action rule
        # Otherwise wander in a random direction
        return ["UP", "DOWN", "LEFT", "RIGHT"][rng.integers(0, 4)]
    # Return the decision function
    return act

# Compare both agents
for name, fac in [("random", random_agent), ("reflex", reflex_agent)]:
    # Scores on five seeded layouts (index 0 = score)
    sc = [run_episode(fac, s)[0] for s in range(5)]
    # Print the scores
    print(f"{name:<7} first 5 scores: {sc}")

## Task A1 — Build a **model-based** agent
A simple-reflex agent cannot know *when it is finished*: it keeps wandering (−1 per move) even after the grid is clean. A **model-based** agent keeps **internal state** about the world.

Your agent must sweep the grid in a *snake* pattern (right along row 0, down one row, left along row 1, …). It knows the grid size `n` and keeps an **internal model of its own position** by *dead reckoning* (it knows where it is because it remembers what it did). It must `CLEAN` whenever the percept says `dirty`, and switch to `NOOP` after the last cell of the last row.

*Hint:* internal state = `r`, `c` (estimated row/column) and `dir` (+1 = moving right, −1 = left). At the end of a row (`c == n-1` going right, or `c == 0` going left) move `DOWN` once and flip `dir`.

In [ ]:
# TODO(Student): implement model_based_agent(rng) → an act(percept) function WITH internal state
# TODO: an agent with memory (internal model of the world)
def model_based_agent(rng, n=5):
    # The internal model: row, column, and sweep direction
    state = {"r": 0, "c": 0, "dir": +1}          # internal model: where am I, which way am I sweeping?
    # The decision function the environment will call
    def act(p):
        # 1) clean if the percept says dirty
        # 2) if I am at the end of the last row → "NOOP"
        # 3) if I am at the end of a row → "DOWN"; update r and flip dir
        # 4) otherwise step RIGHT/LEFT according to dir; update c
        # Placeholder so the cell runs; replace with your decision rules
        return "NOOP"
    # Return the decision function
    return act

In [ ]:
#@title ✅ Auto-check for Task A1 (the model-based agent should clean everything and then stop)
# Run the model-based agent once and print the outcome
sc, left = run_episode(model_based_agent, seed=0)
check("leaves no dirt behind (seed 0)", lambda: left == 0, "sweep every cell exactly once")
check("score ≥ 15 on seed 0",         lambda: sc >= 15)
# Reflex agent scores on 30 layouts
sc_ref = [run_episode(reflex_agent, s)[0] for s in range(30)]
# Model-based agent scores on the same 30 layouts (paired design)
sc_mb  = [run_episode(model_based_agent, s)[0] for s in range(30)]
check("beats the reflex agent on average over 30 seeds", lambda: np.mean(sc_mb) > np.mean(sc_ref) + 50, "should be ~70 points better")

## Part B — Is it *rational*? Evaluate with statistics
Rationality is about **expected** performance, so one run proves nothing. Run each agent on the **same 100 dirt layouts** (seeds 0–99), report mean, sd and 95% CI, and use the **paired Wilcoxon signed-rank test** (non-parametric: scores are not normal).

In [ ]:
#@title 📊 Compare the three agents (100 paired episodes)
# 100 layouts shared by all agents for a fair comparison
SEEDS = range(100)
# Score every agent on every layout
scores = {name: [run_episode(fac, s)[0] for s in SEEDS]
          for name, fac in [("random", random_agent), ("reflex", reflex_agent), ("model-based", model_based_agent)]}
# Report each agent's mean, SD and 95% CI
for name, x in scores.items():
    # Summary statistics line
    report(name, x)
# Paired Wilcoxon test: reflex vs random
print("\nreflex vs random      p =", f"{paired_wilcoxon(scores['reflex'], scores['random']):.2e}")
# Paired Wilcoxon test: model-based vs reflex
print("model-based vs reflex p =", f"{paired_wilcoxon(scores['model-based'], scores['reflex']):.2e}")

# Plotting library
import matplotlib.pyplot as plt
# Create a figure of the chosen size
plt.figure(figsize=(6, 3.2))
# Overlay one histogram per agent
for name, x in scores.items(): plt.hist(x, bins=15, alpha=0.5, label=name)
# Axis labels, legend and title
plt.xlabel("episode score (higher is better)"); plt.ylabel("frequency"); plt.legend(); plt.title("Score distributions, n = 100 layouts")
# Tidy the layout and show the figure
plt.tight_layout(); plt.show()
check("reflex > random (p < 0.001)",          lambda: np.mean(scores["reflex"]) > np.mean(scores["random"]) and paired_wilcoxon(scores["reflex"], scores["random"]) < 1e-3)
check("model-based > reflex (p < 0.001)",     lambda: np.mean(scores["model-based"]) > np.mean(scores["reflex"]) and paired_wilcoxon(scores["model-based"], scores["reflex"]) < 1e-3)

**Discuss.** (1) Which agent is *rational* under this performance measure? (2) Change the measure so that moving costs 0 and `NOOP` costs −1. Which agent becomes rational? *(Rationality is relative to the performance measure.)* (3) The model-based agent assumes dirt never reappears. What happens in a **dynamic** environment, and what kind of agent would you need?

## Task B1 — Classify task environments
Complete the table with `True/False/str` values. Properties: *fully observable*, *deterministic*, *episodic*, *static*, *discrete*, *single-agent*.

In [ ]:
# TODO(Student): classify each task environment (fill in the six properties)
# PEAS descriptions: Performance, Environment, Actuators, Sensors for each task
ENVS = {
    "vacuum world (this lab)": dict(observable=None, deterministic=None, episodic=None, static=None, discrete=None, single_agent=None),
    "chess (no clock)":        dict(observable=None, deterministic=None, episodic=None, static=None, discrete=None, single_agent=None),
    "driving in Lagos traffic":dict(observable=None, deterministic=None, episodic=None, static=None, discrete=None, single_agent=None),
    "classifying spam e-mails":dict(observable=None, deterministic=None, episodic=None, static=None, discrete=None, single_agent=None),
    "customer-service chatbot":dict(observable=None, deterministic=None, episodic=None, static=None, discrete=None, single_agent=None),
    "autonomous vehicle (Lagos)":dict(observable=None, deterministic=None, episodic=None, static=None, discrete=None, single_agent=None),
    "medical diagnosis system":dict(observable=None, deterministic=None, episodic=None, static=None, discrete=None, single_agent=None),
}

In [ ]:
#@title ✅ Auto-check for Task B1
check("all cells answered (True/False)", lambda: all(isinstance(v, bool) for e in ENVS.values() for v in e.values()))
# Pick one environment to study
d = ENVS["driving in Lagos traffic"]
check("driving is stochastic, dynamic, multi-agent", lambda: d["deterministic"] is False and d["static"] is False and d["single_agent"] is False)
check("spam classification is episodic", lambda: ENVS["classifying spam e-mails"]["episodic"] is True)
check("autonomous vehicle matches driving in Lagos", lambda: ENVS["autonomous vehicle (Lagos)"] == ENVS["driving in Lagos traffic"])
check("chess is sequential", lambda: ENVS["chess (no clock)"]["episodic"] is False)

**Discuss (pairs, 5 min).** The chatbot, vehicle and diagnosis rows are *debatable*: is the chatbot's world static while it composes a reply? Is a patient's illness static during a consultation? Defend each answer in one sentence; there can be more than one defensible answer when you state your assumptions.

## Task B3 — Score the agent and update its belief (note §2.2 and §2.5)
Two ideas from the class note:

**(a) Performance measure.** $V=\sum_t \gamma^t R(s_t,a_t)$. For the vacuum, reward $+1$ per square cleaned and $-0.1$ per move ($\gamma=1$): cleaning 3 squares in 8 moves scores $3-0.8=2.2$; in 5 moves, $2.5$.

**(b) Belief update.** The agent keeps a probability for each state it may be in and updates it with Bayes' rule after each percept. With no state change,
$$b'(s)=\frac{P(p\mid s)\,b(s)}{\sum_{s'}P(p\mid s')\,b(s')} .$$
Example 2.2 in the note: $b=(0.5,0.5)$, $P(\text{dirt}\mid s_1)=0.9$, $P(\text{dirt}\mid s_2)=0.2$ gives $b'=(0.818,0.182)$.

In [ ]:
# TODO(Student): implement the two functions
def episode_value(cleaned, moves, move_cost=0.1, gamma=1.0):
    """Value of an episode when every move costs `move_cost` (gamma = 1 for this exercise)."""
    # Placeholder; replace with your performance measure
    return None

def update_belief(belief, likelihood):
    """belief: list of P(state); likelihood: list of P(percept | state). Return the normalised posterior."""
    # Placeholder; replace with your performance measure
    return None

In [ ]:
#@title ✅ Auto-check for Task B3
check("episode value 3 cleaned, 8 moves = 2.2", lambda: abs(episode_value(3, 8) - 2.2) < 1e-9)
check("fewer moves score higher",               lambda: episode_value(3, 5) > episode_value(3, 8))
# Update a 50/50 belief after evidence favouring hypothesis 1
b = update_belief([0.5, 0.5], [0.9, 0.2])
check("belief after 'dirt' = (0.818, 0.182)",   lambda: abs(b[0] - 0.818) < 1e-3 and abs(sum(b) - 1) < 1e-9)
# Update after evidence favouring hypothesis 2
b2 = update_belief([0.5, 0.5], [0.1, 0.8])
check("belief after 'no dirt' (Exercise 2.1) = 0.111", lambda: abs(b2[0] - 0.111) < 1e-3)
# Second update from the posterior (only if the first one is done)
b3 = update_belief(b, [0.9, 0.2]) if b else None
check("a second noisy reading raises confidence", lambda: b3[0] > b[0])
# Print the beliefs before and after
if b and b3: print("beliefs:", [round(x, 3) for x in b], "→", [round(x, 3) for x in b3])

## Task B2 — Agent types and architectures
Classify each system as **simple reflex / model-based / goal-based / utility-based / learning**, and as **reactive / deliberative / hybrid** (architecture):

1. A prepaid-meter that cuts power when credit = 0.
2. Google Maps re-routing around a jam.
3. A spam filter that updates its weights every night.
4. A campus robot guide that plans a route (deliberative layer) but swerves around students (reactive layer).

## 📝 Assignment (5 marks, individual, formative)
1. *(2)* Write a **PEAS** description for a task of your own (e.g. a campus-shuttle dispatcher) and classify its environment on all six properties with a one-line reason each.
2. *(2)* Add a **fourth agent** (for example a reflex agent that sweeps rows) and compare it with the model-based agent over 100 seeded layouts: mean ± sd, 95% CI and a paired Wilcoxon *p*.
3. *(1)* Do one Bayesian belief update with numbers of your own and explain the posterior in a sentence a non-technical person would follow.

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
- Which agent is "intelligent": the one with the highest score or the one with the simplest code? Defend your answer using the performance measure.
- Which environment property made the model-based agent necessary here?

```text
Write your reflection here…
```

## Extensions
- Add **utility**: dirt in some cells is worth 20. Re-design the sweep to visit high-value cells first (utility-based agent).
- Make dirt re-appear with probability 0.01 per cell per step and re-evaluate all agents.

In [ ]:
checks_summary()  # how many auto-checks have you passed?